In [4]:
import pandas as pd
import numpy as np

In [5]:
df= pd.read_csv('D:/Sets of excel and bi/train.csv')

In [6]:
df.shape

(9800, 18)

In [7]:
list(df.columns)

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales']

In [8]:
df.dtypes

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code      float64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
dtype: object

In [9]:
##STANDARDISE COLUMN NAMES(easy to write,and remember for dax)
## ["Name ", "Total Sales", "Order-Date"] into ["name", "total_sales", "order_date"]
## if dataset columns name include special characters,dots,unnessary space,mixed casing.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]", "_", regex=True)  # sab special chars hata dega
)

In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [11]:
##date parsing (converting dates wly columns into standard date format)
for col in ["order_date", "ship_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], dayfirst=False, errors="coerce")

In [12]:
##new columns (feature Engineering)
df["order_year"]    = df["order_date"].dt.year
df["order_month"]   = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["order_weekday"] = df["order_date"].dt.day_name()
df["ship_days"]     = (df["ship_date"] - df["order_date"]).dt.days

In [13]:
print("Checking nulls...")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

Checking nulls...
order_date       5841
ship_date        5985
postal_code        11
order_year       5841
order_month      5841
order_quarter    5841
order_weekday    5841
ship_days        7124
dtype: int64


In [14]:
# Drop rows where key financial fields are missing
df.dropna(subset=["sales", "order_date"], inplace=True)

In [15]:
# Fill postal code nulls with 'Unknown'
if "postal_code" in df.columns:
    df["postal_code"] = df["postal_code"].fillna("Unknown").astype(str)

In [16]:
#TYPE FIXES 
numeric_cols = ["sales"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

In [17]:
print("Checking nulls...")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

Checking nulls...
ship_date    1283
ship_days    1283
dtype: int64


In [18]:
before = len(df)
if "order_id" in df.columns and "product_id" in df.columns:
    df.drop_duplicates(subset=["order_id", "product_id"], inplace=True)
else:
    df.drop_duplicates(inplace=True)
print(f"\n Removed {before - len(df)} duplicate rows")


 Removed 3 duplicate rows


In [19]:
if "category" in df.columns:
    df["category"] = df["category"].str.strip().str.title()
if "region" in df.columns:
    df["region"] = df["region"].str.strip().str.title()
if "segment" in df.columns:
    df["segment"] = df["segment"].str.strip().str.title()

In [20]:
## outlier removal
q_low  = df["sales"].quantile(0.01)
q_high = df["sales"].quantile(0.99)
df["sales_outlier_flag"] = (
    (df["sales"] < q_low) | (df["sales"] > q_high)
).astype(int)

In [21]:
# MONTHLY SUMMARY TABLE 
monthly = (
    df.groupby(["order_year", "order_month", "category", "region"])
    .agg(
        total_sales = ("sales", "sum"),
        num_orders  = ("order_id", "count"),
    )
    .reset_index()
)

In [22]:
df['order_status'] = df['ship_date'].apply(lambda x: 'Not Shipped' if pd.isnull(x) else 'Shipped')

In [23]:
print("\n Saving outputs...")
with pd.ExcelWriter("retail_cleaned.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Transactions",    index=False)
    monthly.to_excel(writer, sheet_name="Monthly_Summary", index=False)

print("\n Done! File saved: retail_cleaned.xlsx")
print(f"   Final rows (Transactions)  : {len(df):,}")
print(f"   Final rows (Monthly)       : {len(monthly):,}")


 Saving outputs...

 Done! File saved: retail_cleaned.xlsx
   Final rows (Transactions)  : 3,956
   Final rows (Monthly)       : 556
